# Indian Banknote Damage Detection — Dataset Audit

This notebook audits the folder structure and image distribution before training.

## Important dataset note

The recommended Mendeley **spoilt banknote** dataset contains **5,125 damaged images** across eight classes based on denomination and old/new note type. It does **not** directly provide labels such as `Torn`, `Soiled`, `Crumpled`, or `Missing Corner`.

Therefore:
- use the original dataset for **spoilt-note / denomination-oriented experiments**;
- create separate verified annotations before training a fine-grained damage-severity model.


In [ ]:
from pathlib import Path
from collections import Counter
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

DATA_DIR = Path("data")
IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

images = [p for p in DATA_DIR.rglob("*") if p.suffix.lower() in IMAGE_EXTS]

print("Total images found:", len(images))
print("Data directory:", DATA_DIR.resolve())


In [ ]:
rows = []
for p in images:
    # Assumes parent folder is class name.
    rows.append({
        "path": str(p),
        "class_name": p.parent.name,
        "extension": p.suffix.lower()
    })

audit = pd.DataFrame(rows)

if audit.empty:
    print("No images found yet. See data/README.md for dataset sources.")
else:
    display(audit.head())
    display(audit["class_name"].value_counts().to_frame("images"))


In [ ]:
if not audit.empty:
    counts = audit["class_name"].value_counts().sort_values()
    plt.figure(figsize=(10,6))
    plt.barh(counts.index, counts.values)
    plt.title("Images per Class")
    plt.xlabel("Number of images")
    plt.tight_layout()
    plt.show()


In [ ]:
# Inspect a few image dimensions
sample_rows = []
for p in images[:100]:
    try:
        with Image.open(p) as im:
            sample_rows.append({
                "path": str(p),
                "width": im.width,
                "height": im.height,
                "mode": im.mode
            })
    except Exception as e:
        sample_rows.append({"path": str(p), "error": str(e)})

dims = pd.DataFrame(sample_rows)
if not dims.empty:
    display(dims.head(20))


## What to verify before training

1. Are the class names correct?
2. Are duplicate images present?
3. Are train/validation/test splits independent?
4. Are the classes reasonably balanced?
5. Do folder names describe denomination or actual damage severity?
6. Are there enough examples per damage class before claiming a severity classifier?
